#### Business Understanding
**Problem Statement:** Access to timely and reliable healthcare remains a major challenge, especially in regions with:
- Limited doctor-to-patient ratios 
- Long waiting times at clinics 
- High cost of consultations 
- Low health literacy among patients 
Many individuals:
- Ignore early symptoms 
- Misinterpret health information online 
- Delay seeking care until conditions worsen 
At the same time, general-purpose AI systems often:
- Provide unsafe or inaccurate medical advice 
- Lack grounding in trusted clinical guidelines 
- Do not assess urgency (triage) effectively 

This creates a critical gap: There is no widely accessible, safe, and intelligent system that can guide patients on what to do next based on their symptoms, without attempting full diagnosis or unsafe prescriptions.
Project Description:  Smart Doctor is a RAG-based AI Medical Triage & Advisory Assistant designed to:
- Interact with patients via text (and later voice) 
- Collect symptoms through structured conversation 
- Ask intelligent follow-up questions 
- Assess the urgency of the condition (triage) 
- Provide safe, evidence-based health advice 
- Recommend next steps (self-care, clinic visit, emergency care) 
- Store patient interaction data securely for future reference 

The system leverages:
- Large Language Models (LLMs) for conversation 
- Retrieval-Augmented Generation (RAG) for grounded medical knowledge 
- Rule-based logic for safe triage decision-making 

Importantly, the system:
Does NOT diagnose or prescribe medication, but instead supports decision-making and early intervention.
 Project Measurables (KPIs):
AI Performance Metrics

**1. Symptom Extraction Accuracy**
- Percentage of correctly identified symptoms from user input 
- Target: ≥ 85% accuracy 

**2. Triage Classification Accuracy**
- Agreement with medical guidelines or expert validation 
- Target: ≥ 90% for rule-based scenarios

**3. Response Grounding Score (RAG Quality)**
- Percentage of responses backed by retrieved medical sources 
- Target: ≥ 95% grounded responses 
 B. Safety Metrics

**4. Emergency Detection Recall**
- % of true emergencies correctly flagged 
- Target: ~100% (very critical) 

**5. Hallucination Rate**
- % of responses containing unsupported claims 
- Target: < 5% 

**6. Unsafe Recommendation Rate**

- Instances of: 
    - Prescriptions 
    - Confident diagnosis 
    - Target: 0% 
    
C. User Experience Metrics

**7. Conversation Completion Rate**
- Percentage of users who finish triage flow 
- Target: ≥ 80% 

**8. User Satisfaction Score**
- Feedback rating (1–5 scale) 
- Target: ≥ 4.0 

 D. System Metrics

**9. Response Time**
- Time taken to generate response 
- Target: < 3 seconds 

**10. System Reliability**
- Uptime / error rate 
- Target: ≥ 99% uptime 


In [12]:
# Import all important libraries
import os
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import json
from urllib.parse import urljoin
import requests
from langchain.chat_models import init_chat_model
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PDFMinerLoader
from langchain_google_genai import GoogleGenerativeAI
from google import genai
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
hugging_face_token = os.getenv("HUGGINGFACE_TOKEN")



#### Load Documents

In [2]:
# Scrape Disease Fact Sheets from WHO website
BASE_URL = "https://www.who.int/"
URL = "https://www.who.int/news-room/fact-sheets"

# get response from the URL and parse it using BeautifulSoup
response = requests.get(URL)
soup = BeautifulSoup(response.text,'html.parser')
links = []

# # Extracting all the links of the disease fact sheets
for a in soup.find_all('a', href = True):
    href = a['href']
    if "/news-room/fact-sheets/detail/" in href:
        full_url = urljoin(BASE_URL, href)
        title = a.text.strip()
        links.append((title, full_url))

In [4]:
# Scrape the content of each disease fact sheet 
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

def scrape_content(url):
    response = requests.get(url, headers = headers, timeout=10)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Extract title
    title = soup.find('h1').text.strip() if soup.find('h1') else 'No title found'
    
    # Scrape the content of the page
    content = { }
    content_div = soup.find('article', class_= 'sf-detail-body-wrapper')

    # print(content_div.prettify())
    if content_div:
        # current_heading = None
        required_sections = [
                    'Overview',
                    'Signs and symptoms',
                    'Causes',
                    'Treatment and prevention',
                    'Self-care',
                    'WHO response'
                ]
        current_heading = None
        for tag in content_div.find_all(['h2', 'p','li']):
            # Content headings
            if tag.name == 'h2':
                heading = tag.get_text(strip=True)
                if heading in required_sections:
                    current_heading = heading
                    content[current_heading] = []
                else:
                    current_heading = None
            # Content paragraphs
            elif tag.name == 'p':
                text = tag.get_text(" ",strip=True)
                if text and current_heading:
                    content[current_heading].append(text)
            # Content list items
            elif tag.name == 'li':
                text = tag.get_text(" ",strip=True)
                if text and current_heading:
                    content[current_heading].append(f" - {text}")

    data = {
        "title": title,
        "content": content
    }

    return data

scraped_data = []
for name, link in links:
    data = scrape_content(link)
    scraped_data.append(data)


#### Save Scraped Data in A CSV 

In [6]:
# Save the scraped content in a JSON file
with open('../datasets/disease_fact_sheets.json', 'w', encoding='utf-8') as f:
    json.dump(scraped_data, f, indent=4, ensure_ascii=False)
    

### Load PDF Documents and Excel Documents 

In [2]:
medquad_df = pd.read_csv('../datasets/medquad.csv')

medquad_df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


#### Convert MedQuad CSV Data to Documents

In [3]:

medquad_docs = []

for index, row in medquad_df.iterrows():
    text = f"""
            question: {row['question']}

        
            answer:  {row['answer']}
    """
    medquad_docs.append(
        Document(
            page_content=text,
            metadata = {
                'source' : row['source'],
                'focus_area': row['focus_area'],
                'type': 'qa_dataset'
            }

            
        )
    )

print(medquad_docs[0])

page_content='
            question: What is (are) Glaucoma ?


            answer:  Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye 

In [4]:
disease_and_symptoms_df = pd.read_csv('../datasets/Diseases_and_Symptoms_dataset.csv')
disease_and_symptoms_df.head()

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,sneezing,leg weakness,hysterical behavior,arm lump or mass,bleeding gums,pain in gums,diaper rash,hesitancy,back stiffness or tightness,low urine output
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [5]:
symptoms_docs = []
symptom_columns = disease_and_symptoms_df[1:] #skip disease columns
for index, row in disease_and_symptoms_df.iterrows():
    disease_name = row['diseases']
    
    symptoms = [ ]
    for symptom in symptom_columns:
        if row[symptom] == 1:
            symptoms.append(symptom)

    text = f"""
        Disease: {disease_name}

    Symptoms:
            {
                ', '.join(symptoms)
            }
            """

    symptoms_docs.append(
            Document (
            page_content = text,
            metadata={
                'disease_name':disease_name,
                'type':'symptoms_dataset'
            }
            )
        )

# print(symptoms_docs[0].page_content)
 

In [6]:
# Load the Ghana Health Service PDF document
ghs_pdf = PDFMinerLoader('../datasets/GHANA-STG-2017-1.pdf')
ghs_data = ghs_pdf.load()
ghs_data


[Document(metadata={'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 2017 (Windows)', 'creationdate': '2017-08-23T12:37:18+01:00', 'moddate': '2020-07-09T14:43:23+00:00', 'trapped': 'False', 'total_pages': 708, 'source': '../datasets/GHANA-STG-2017-1.pdf'}, page_content='Republic of Ghana\n\nMinistry of Health \nGhana National Drugs Programme \n(GNDP)\n\nStandard Treatment Guidelines\n\nMinistry of Health\nSeventh Edition (7th), 2017\n\ni\n\x0c© 2017 Ministry of Health (GNDP) Ghana\nAll rights reserved. No part of this publication may be reproduced, stored \nin  any  retrieval  system,  or  transmitted  in  any  form  or  by  any  means, \nelectronic,  mechanical,  photocopying,  recording  and/or  otherwise, \nwithout prior written permission of the Ministry of Health.\n\nEssential Drugs List & National Formulary with Therapeutic Guidelines, \n1st Edition, 1988\nEssential Drugs List & National Formulary with Therapeutic Guidelines, \n2nd Edition, 1993\nEssential Dru

#### Convert WHO from JSON to docs

In [7]:
# load JSON file containing WHO disease fact sheets
who_df= pd.read_json('../datasets/disease_fact_sheets.json')

who_docs = []
for index, row in who_df.iterrows():
    disease_name = row["title"]
    contents = row["content"]
    # Get individual contents from the content
    
    text = f"""
        disease_name : {disease_name}
        overview : {' '.join(contents.get('Overview', []))}
        symptoms : {' '.join(contents.get('Signs and symptoms', []))}
        causes : {' '.join(contents.get('Causes', []))}
        Treatment: {' '.join(contents.get('Treatment and prevention', []))}
        Self-care: {' '.join(contents.get('Self-care', []))}
        WHO response: {' '.join(contents.get('WHO response', []))}

        
    
    """
    
    who_docs.append(
        Document(
            page_content=text,
            metadata={
                'source': 'WHO',
                'type': 'who_disease_fact_sheet'
            }
        )
    )


#### Combine all the Docs

In [8]:
all_docs = ( who_docs + ghs_data + medquad_docs + symptoms_docs)

print(len(all_docs))

112741


### Chunk Documents 

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 700,
    chunk_overlap = 150
)

split_docs = splitter.split_documents(all_docs)
print(len(split_docs))

156472


#### Embed Text with HuggingFace Embeddings

In [2]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name = model_name, model_kwargs={"device": "cpu"})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1983.97it/s]


#### Store Embeddings In FAISS 

In [11]:
vectorstore = FAISS.from_documents(
    split_docs,
    embeddings
)


In [12]:
# Save embeddings
vectorstore.save_local("medical_rag_embeddings")

#### Load Local Embeddings

In [3]:
# Retrieve embeddings
vectorstore = FAISS.load_local("medical_rag_embeddings", embeddings, allow_dangerous_deserialization=True)

#### Create a Retriever

In [4]:
# Create a retriever from the vectorstore
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Use Maximal Marginal Relevance for retrieval
    search_kwargs = {'k':5,
                     "fetch_k": 20}
)

#### Define GEMINI Model

In [7]:
llm = GoogleGenerativeAI(
    model= "gemini-2.5-flash",
    temperature=0,
    api_key = gemini_api_key
)

#### Create the Prompt

In [10]:
# create prompt template for the RAG model
prompt = PromptTemplate.from_template("""
You are Smart Pocket Doctor.

Use ONLY the provided context to answer.

If the answer is not in the context, say:
"I could not find enough information in the medical database. Kindly consult a healthcare professional for accurate guidance."

Context:
{context}

Question:
{question}

Answer:
""") 


### Build the RAG Chain

In [13]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),

    }
    |prompt
    |llm
    |StrOutputParser()
)

In [20]:
# Ask a question to the RAG model
query = "How can I treat malaria?"
response = rag_chain.invoke(query)
print(response)
docs = retriever.invoke(query)
for doc in docs:
    print(doc.metadata)


Malaria can be treated with medicines.

For severe malaria, parenteral antimalarials are recommended for a minimum of 24 hours, followed by oral medication once the patient can tolerate it. Recommended follow-on treatments include Artemisinin Combination Therapy (ACTs) and Quinine + clindamycin.

For uncomplicated malaria, Artemisinin Combination Therapy (ACT) is currently recommended to prevent the development of drug resistance. An example of an ACT regimen provided is Artesunate + Amodiaquine co-blistered formulation.
{'source': 'WHO', 'type': 'who_disease_fact_sheet'}
{'source': 'GARD', 'focus_area': 'Zika virus infection', 'type': 'qa_dataset'}
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 2017 (Windows)', 'creationdate': '2017-08-23T12:37:18+01:00', 'moddate': '2020-07-09T14:43:23+00:00', 'trapped': 'False', 'total_pages': 708, 'source': '../datasets/GHANA-STG-2017-1.pdf'}
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 2017 (Windows)',

In [5]:
# Create LLM client
client = genai.Client(api_key=gemini_api_key)
# Define the model name
model_name = "gemini-2.5-flash"
# Create a class for the Gemini model
class GEMINI_MODEL:
    # Initialize the class with the client and model name
    def __init__(self, client, model_name):
        self.client = client
        self.model_name = model_name
# Define a method to generate content using the Gemini model
    def generate_content(self, contents):
        response = self.client.models.generate_content(
            model = self.model_name,
            contents = contents
        )
        return response

model = GEMINI_MODEL(client, model_name)

#### Create Embeddings